# 5.7 — Train MuscleMimic from scratch with FastTD3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MyoHub/myosuite/blob/tutorial/fasttd3-directional-scratch/tutorials/5.7_FastTD3_Directional_From_Scratch.ipynb)

Tutorial 5.3 trains a MuscleMimic walker with **on-policy PPO**. This notebook trains the same full-body model **from scratch** with **off-policy FastTD3** (Seo et al., 2025): no motion clip, no Hugging Face token, no behaviour cloning.

The task is `myoMimicFullbody-v0` (354 muscles, random site targets). FastTD3's actor emits `tanh` actions in `[-1, 1]`; the env wrapper maps them to muscle activations in `[0, 1]`.

## Flow

1. Open the env and inspect observation / action sizes.
2. Smoke-train on CPU (a few steps, tiny nets).
3. Optionally train on GPU with the mjlab Warp twin.

The CPU smoke is the default. A 50k-step T4 run is a demo, not a finished walker.

Local install, from a MyoSuite clone:

```bash
pip install -e ".[musclemimic]"
pip install -e training/fasttd3
```

Colab: run **5.7.0** once, then continue.

> **Citation:** FastTD3 — [younggyoseo/FastTD3](https://github.com/younggyoseo/FastTD3) (Seo et al., 2025, MIT). MuscleMimic — [Li et al., 2026](https://arxiv.org/abs/2603.25544). This notebook does not download AMASS/KIT clips. See the MyoSuite *References* page.

## 5.7.0 — Colab install

Skip this cell if you already installed MyoSuite locally. It only clones the repo and installs the MuscleMimic extra plus the FastTD3 trainer.

In [ ]:
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    %cd /content
    if not Path("/content/myosuite/myosuite").exists():
        !git clone --depth 1 -b tutorial/fasttd3-directional-scratch https://github.com/MyoHub/myosuite.git
    %cd /content/myosuite
    %pip install -q --upgrade-strategy only-if-needed -e ".[musclemimic]"
    %pip install -q --upgrade-strategy only-if-needed -e "./training/fasttd3"
else:
    print("Local runtime — skip Colab install.")

## 5.7.1 — Config

`GPU_TRAIN` stays off until the runtime has a GPU (Colab: Runtime → Change runtime type → T4).

In [ ]:
from pathlib import Path
import os
import sys

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ.setdefault("JAX_PLATFORMS", "cpu")

ENV_ID = "myoMimicFullbody-v0"
GPU_TRAIN = False
TRAIN_STEPS = 50_000
N_ENVS = 64
OUT = Path("runs/fasttd3")

for d in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (d / "myosuite" / "__init__.py").is_file():
        sys.path[:0] = [str(d), str(d / "training" / "fasttd3")]
        break

print(f"{ENV_ID}  GPU_TRAIN={GPU_TRAIN}  steps={TRAIN_STEPS}  envs={N_ENVS}")

## 5.7.2 — The environment

`myoMimicFullbody-v0` is the MuscleMimic full-body model (`musclemimic_models` 1.0.6) with random site targets — the same skeleton as ChaseTag and tutorial 5.4, without gated KIT clips.

Observation is about 684 values (qpos, qvel, activations, sites, target). Action is 354 muscle excitations in `[0, 1]`.

In [ ]:
from myosuite.core.registry import make_env

env = make_env(ENV_ID)
obs, _ = env.reset(seed=0)
print("observation", obs.shape)
print("action     ", env.action_space.shape, env.action_space.low.min(), "…", env.action_space.high.max())
env.close()

## 5.7.3 — FastTD3

The trainer lives under `training/fasttd3/` so method-specific dependencies stay out of the installable `myosuite` package. It uses the published FastTD3 actor, distributional critic, and replay buffer.

```python
from myosuite_fasttd3 import TrainArgs, train
```

AMP stays off on T4 (no bf16; fp16 overflows on this observation). Critic support is `[-250, 250]`.

## 5.7.4 — CPU smoke

Two environments, eight steps, small networks. This checks the loop; it will not learn to walk. You should see `backend=cpu`, `amp=False`, and a saved `smoke.pt`.

In [ ]:
from myosuite_fasttd3 import TrainArgs, train

ckpt = train(TrainArgs(env_id=ENV_ID, smoke=True, backend="cpu", device="cpu", out=OUT))
print("checkpoint:", ckpt)

## 5.7.5 — GPU training (optional)

Same env id, mjlab Warp backend. On Colab, switch to a **T4** runtime, rerun from 5.7.0, set `GPU_TRAIN = True`, then run this cell.

Needs `mjlab`, `mujoco-warp`, and `warp-lang` (already on a local `[mjlab]` install). Log fields: `qloss`, `q` (should stay in roughly `[-250, 250]`), `pi`, `r` (step reward, about 0–1 here).

In [ ]:
import torch
from myosuite_fasttd3 import TrainArgs, train

if not GPU_TRAIN:
    print("GPU_TRAIN is False — skip. Enable it on a T4 runtime after the CPU smoke.")
elif not torch.cuda.is_available():
    print("No GPU in this session. Colab: Runtime → Change runtime type → T4, then rerun.")
else:
    import importlib.util

    if importlib.util.find_spec("mjlab") is None:
        print("Install the Warp stack first: pip install mjlab mujoco-warp warp-lang")
    else:
        ckpt = train(
            TrainArgs(
                env_id=ENV_ID,
                backend="mjlab",
                num_envs=N_ENVS,
                total_timesteps=TRAIN_STEPS,
                out=OUT,
            )
        )
        print("checkpoint:", ckpt)

## 5.7.6 — Next

- Larger GPU run: `train(TrainArgs(env_id=ENV_ID, backend="mjlab", num_envs=1024, total_timesteps=150_000))`
- Heading-conditioned walking with a teacher: tutorial **5.4**
- ChaseTag submission: tutorial **5.6**
- Trainer notes: `training/fasttd3/README.md`